# Async aiohttp Fake News Scraper



In [1]:
# Install only if needed
# !pip install aiohttp newspaper3k beautifulsoup4 tqdm pandas nltk scikit-learn lxml_html_clean

In [2]:
import os, re, time, hashlib, random, asyncio
import aiohttp
import pandas as pd
import nltk

from bs4 import BeautifulSoup
from newspaper import Article
from tqdm.asyncio import tqdm_asyncio
from tqdm import tqdm
from urllib.parse import urljoin
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

OUTPUT_FOLDER = "generated_fake_news_dataset"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

TARGET_TOTAL_RECORDS = 60001
TARGET_PER_CLASS = 30000

# Async speed settings. If websites block you or too many requests fail, reduce these values.
URL_COLLECTION_CONCURRENCY = 50
ARTICLE_EXTRACTION_CONCURRENCY = 80
REQUEST_TIMEOUT_SECONDS = 10
MAX_URLS_PER_ROUND = 3000
MAX_TOTAL_CANDIDATE_URLS = 100000
CHECKPOINT_EVERY = 500

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

REAL_SOURCES = {
    "BBC": [
        "https://www.bbc.com/news",
        "https://feeds.bbci.co.uk/news/rss.xml",
        "https://www.bbc.com/sitemaps/https-sitemap-com-news-1.xml"
    ],
    "NPR": [
        "https://www.npr.org",
        "https://feeds.npr.org/1001/rss.xml",
        "https://www.npr.org/sitemaps/sitemap.xml"
    ],
    "Guardian": [
        "https://www.theguardian.com/international",
        "https://www.theguardian.com/world/rss",
        "https://www.theguardian.com/sitemaps/news.xml"
    ],
    "AP": [
        "https://apnews.com",
        "https://apnews.com/hub/ap-top-news"
    ]
}

FAKE_FACTCHECK_SOURCES = {
    "Snopes": [
        "https://www.snopes.com",
        "https://www.snopes.com/sitemap.xml"
    ],
    "PolitiFact": [
        "https://www.politifact.com",
        "https://www.politifact.com/sitemap.xml"
    ],
    "FullFact": [
        "https://fullfact.org",
        "https://fullfact.org/sitemap.xml"
    ],
    "LeadStories": [
        "https://leadstories.com",
        "https://leadstories.com/sitemap.xml"
    ]
}

BAD_LINK_KEYWORDS = [
    "login", "subscribe", "privacy", "terms", "contact",
    "about", "advertise", "video", "podcast", "tag", "author"
]

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)


def get_text_hash(text):
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()


async def fetch_url(session, url, semaphore):
    """Fast async HTML downloader."""
    try:
        async with semaphore:
            async with session.get(url, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                if response.status == 200:
                    return await response.text(errors="ignore")
    except Exception:
        return None
    return None


def extract_links_from_html(base_url, html):
    links = set()
    if not html:
        return links

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup.find_all(["a", "loc"]):
        if tag.name == "loc":
            href = tag.get_text(strip=True)
        else:
            href = tag.get("href")

        if not href:
            continue

        full_url = urljoin(base_url, href).split("#")[0]

        if full_url.startswith("http"):
            links.add(full_url)

    return links


async def process_url_for_links(session, semaphore, source_name, url):
    html = await fetch_url(session, url, semaphore)
    links = extract_links_from_html(url, html)
    return source_name, url, links


async def collect_urls_async(source_dict, max_rounds=3):
    all_links = set()
    queue = []

    for source_name, urls in source_dict.items():
        for u in urls:
            queue.append((source_name, u))

    visited = set()
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS)
    connector = aiohttp.TCPConnector(limit=URL_COLLECTION_CONCURRENCY, ssl=False)
    semaphore = asyncio.Semaphore(URL_COLLECTION_CONCURRENCY)

    async with aiohttp.ClientSession(headers=HEADERS, timeout=timeout, connector=connector) as session:
        for round_no in range(max_rounds):
            print(f"\nURL collection round {round_no + 1}")

            current_queue = []
            for source_name, url in queue:
                if url not in visited:
                    visited.add(url)
                    current_queue.append((source_name, url))

            tasks = [
                process_url_for_links(session, semaphore, source_name, url)
                for source_name, url in current_queue
            ]

            new_queue = []

            for result in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
                try:
                    source_name, url, links = await result
                except Exception:
                    continue

                for link in links:
                    lower_link = link.lower()
                    if any(bad in lower_link for bad in BAD_LINK_KEYWORDS):
                        continue

                    all_links.add((source_name, link))

                    if len(all_links) < MAX_TOTAL_CANDIDATE_URLS:
                        new_queue.append((source_name, link))

            queue = new_queue[:MAX_URLS_PER_ROUND]
            print("Total candidate URLs:", len(all_links))

    return list(all_links)


def parse_article_from_html(url, html):
    """Use newspaper3k parsing, but skip its slow internal download step."""
    try:
        if not html:
            return None

        article = Article(url)
        article.set_html(html)
        article.parse()

        title = article.title
        text = article.text
        publish_date = article.publish_date

        if not text or len(text) < 500:
            return None

        return {
            "title": title,
            "text": text,
            "publish_date": publish_date
        }
    except Exception:
        return None


def process_article_html(source_name, url, label, html):
    article_data = parse_article_from_html(url, html)

    if article_data is None:
        return None, {"source": source_name, "url": url}

    text = article_data["text"]
    cleaned = clean_text(text)

    if len(cleaned.split()) < 80:
        return None, {"source": source_name, "url": url}

    text_hash = get_text_hash(cleaned)

    row = {
        "title": article_data["title"],
        "text": text,
        "cleaned_text": cleaned,
        "label": label,
        "source": source_name,
        "url": url,
        "publish_date": article_data["publish_date"],
        "text_hash": text_hash
    }

    return row, None


async def process_article_async(session, semaphore, source_name, url, label):
    html = await fetch_url(session, url, semaphore)
    return process_article_html(source_name, url, label, html)


def save_checkpoint(rows, failed_urls, checkpoint_file, failed_file, label):
    pd.DataFrame(rows).to_csv(checkpoint_file, index=False)
    pd.DataFrame(failed_urls).to_csv(failed_file, index=False)
    print(f"{label} saved checkpoint: {len(rows)} records")


async def scrape_class_async(source_dict, label, target_count):
    print(f"\nCollecting URLs for label: {label}")

    candidate_urls = await collect_urls_async(source_dict, max_rounds=3)
    random.shuffle(candidate_urls)

    print(f"Candidate URLs found for {label}: {len(candidate_urls)}")

    rows = []
    failed_urls = []
    seen_hashes = set()
    seen_urls = set()

    checkpoint_file = os.path.join(OUTPUT_FOLDER, f"{label}_checkpoint.csv")
    failed_file = os.path.join(OUTPUT_FOLDER, f"{label}_failed_urls.csv")

    if os.path.exists(checkpoint_file):
        old_df = pd.read_csv(checkpoint_file)
        rows = old_df.to_dict("records")
        seen_hashes = set(old_df["text_hash"].astype(str)) if "text_hash" in old_df.columns else set()
        seen_urls = set(old_df["url"].astype(str)) if "url" in old_df.columns else set()
        print(f"Loaded checkpoint for {label}: {len(rows)} records")

    candidate_urls = [item for item in candidate_urls if item[1] not in seen_urls]

    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS)
    connector = aiohttp.TCPConnector(limit=ARTICLE_EXTRACTION_CONCURRENCY, ssl=False)
    semaphore = asyncio.Semaphore(ARTICLE_EXTRACTION_CONCURRENCY)

    async with aiohttp.ClientSession(headers=HEADERS, timeout=timeout, connector=connector) as session:
        tasks = [
            process_article_async(session, semaphore, source_name, url, label)
            for source_name, url in candidate_urls
        ]

        for result in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
            if len(rows) >= target_count:
                break

            try:
                row, failed = await result
            except Exception:
                continue

            if failed is not None:
                failed_urls.append(failed)
                continue

            text_hash = row["text_hash"]
            url = row["url"]

            if text_hash in seen_hashes or url in seen_urls:
                continue

            seen_hashes.add(text_hash)
            seen_urls.add(url)
            rows.append(row)

            if len(rows) % CHECKPOINT_EVERY == 0:
                save_checkpoint(rows, failed_urls, checkpoint_file, failed_file, label)

    pd.DataFrame(rows).to_csv(checkpoint_file, index=False)
    pd.DataFrame(failed_urls).to_csv(failed_file, index=False)

    print(f"Final {label} records:", len(rows))
    return pd.DataFrame(rows)


def finalize_and_save_dataset(real_df, fake_df):
    df = pd.concat([real_df, fake_df], ignore_index=True)

    df.drop_duplicates(subset=["text_hash"], inplace=True)
    df.dropna(subset=["text", "cleaned_text"], inplace=True)

    df["word_count"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))
    df["text_length"] = df["cleaned_text"].apply(len)
    df["capital_ratio"] = df["text"].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )
    df["punctuation_count"] = df["text"].apply(
        lambda x: len(re.findall(r"[!?.,]", str(x)))
    )

    df = df[df["word_count"] > 80]

    real_final = df[df["label"] == "real"].sample(
        n=min(TARGET_PER_CLASS, len(df[df["label"] == "real"])),
        random_state=42
    )

    fake_final = df[df["label"] == "fake"].sample(
        n=min(TARGET_PER_CLASS, len(df[df["label"] == "fake"])),
        random_state=42
    )

    df_final = pd.concat([real_final, fake_final], ignore_index=True)

    if len(df_final) >= 60000:
        df_final = df_final.sample(n=60000, random_state=42).reset_index(drop=True)
    else:
        print("WARNING: Less than 60,000 records collected.")
        print("Collected:", len(df_final))

    if len(df_final) < 2 or df_final["label"].nunique() < 2:
        print("WARNING: Not enough data/classes for train_test_split. Saving full dataset only.")
        train_df = pd.DataFrame()
        test_df = pd.DataFrame()
    else:
        test_size = min(12000, max(1, int(len(df_final) * 0.2)))
        train_df, test_df = train_test_split(
            df_final,
            test_size=test_size,
            random_state=42,
            stratify=df_final["label"]
        )

    full_path = os.path.join(OUTPUT_FOLDER, "full_fake_news_dataset_60000.csv")
    train_path = os.path.join(OUTPUT_FOLDER, "fake_news_train_dataset_48000.csv")
    test_path = os.path.join(OUTPUT_FOLDER, "fake_news_test_dataset_12000.csv")
    summary_path = os.path.join(OUTPUT_FOLDER, "scraping_summary.csv")

    df_final.to_csv(full_path, index=False)
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    summary = df_final.groupby(["label", "source"]).size().reset_index(name="records")
    summary.to_csv(summary_path, index=False)

    print("\n====================================")
    print("SCRAPING PIPELINE COMPLETED")
    print("====================================")
    print("Full dataset:", df_final.shape)
    print("Train dataset:", train_df.shape)
    print("Test dataset:", test_df.shape)
    print("\nLabel distribution:")
    print(df_final["label"].value_counts())
    print("\nSource distribution:")
    print(df_final["source"].value_counts())
    print("\nFiles saved in:", OUTPUT_FOLDER)

    return df_final, train_df, test_df




[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ashok\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ashok\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
async def main():
    print("\nSCRAPING REAL NEWS")
    real_df = await scrape_class_async(REAL_SOURCES, "real", TARGET_PER_CLASS + 1000)

    print("\nSCRAPING FACT-CHECK / FAKE CLAIM NEWS")
    fake_df = await scrape_class_async(FAKE_FACTCHECK_SOURCES, "fake", TARGET_PER_CLASS + 1000)

    return finalize_and_save_dataset(real_df, fake_df)


In [4]:
# Run the async scraper in Jupyter
df_final, train_df, test_df = await main()


SCRAPING REAL NEWS


URL collection round 1


  0%|          | 0/11 [00:00<?, ?it/s]C:\Users\ashok\AppData\Local\Temp\ipykernel_2920\3648125741.py:118: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "html.parser")
100%|██████████| 11/11 [00:02<00:00,  4.09it/s]


Total candidate URLs: 1965

URL collection round 2


100%|██████████| 1962/1962 [02:40<00:00, 12.19it/s]


Total candidate URLs: 27180

URL collection round 3


...

fake saved checkpoint: 14000 records


 99%|█████████▉| 62365/63076 [2:29:34<02:46,  4.28it/s]  

fake saved checkpoint: 14500 records


100%|██████████| 63076/63076 [2:31:02<00:00,  6.96it/s]


Final fake records: 14626
Collected: 20771

SCRAPING PIPELINE COMPLETED
Full dataset: (20771, 12)
Train dataset: (16617, 12)
Test dataset: (4154, 12)

Label distribution:
label
fake    14528
real     6243
Name: count, dtype: int64

Source distribution:
source
FullFact       11290
BBC             2119
Snopes          2049
AP              1655
Guardian        1557
PolitiFact       924
NPR              912
LeadStories      265
Name: count, dtype: int64

Files saved in: generated_fake_news_dataset
